# 评分卡建模全流程演示 (Scorecard Full Pipeline)

本notebook演示使用hscredit库完成完整的评分卡建模流程，并输出专业的Excel报告。

参考"建模参考代码.ipynb"的全流程设计，包含：数据准备、特征分箱、WOE编码、特征筛选、模型训练、评分卡生成、规则导出和Excel报告输出。

In [ ]:
# 添加项目路径
import sys
import os
sys.path.append('../')

# 初始化设置
from hscredit.utils import init_setting
init_setting(seed=42)

import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import os
import matplotlib.pyplot as plt

# 设置中文显示
plt.rcParams['font.sans-serif'] = ['SimHei', 'Arial Unicode MS', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False

print("=" * 60)
print("hscredit 评分卡建模全流程")
print("=" * 60)

## Step 1: 数据加载与探索

In [ ]:
# 加载数据
data_path = '/Users/xiaoxi/CodeBuddy/hscredit/hscredit/examples/hscredit.xlsx'
df = pd.read_excel(data_path)

print(f"数据形状: {df.shape}")
print(f"\n列名: {df.columns.tolist()}")
print(f"\n数据类型:")
print(df.dtypes)
print(f"\n缺失值统计:")
print(df.isnull().sum()[df.isnull().sum() > 0])
print(f"\n目标分布:")
print(df['FPD15'].value_counts())
print(f"\n坏账率: {df['FPD15'].mean()*100:.2f}%")

## Step 2: 数据准备

In [ ]:
# 定义目标列和排除列
target_col = 'FPD15'
exclude_cols = ['MOB1', 'MOB2', 'loan_date', 'FPD15', 'SFPD15']

# 获取特征列
feature_cols = [col for col in df.columns if col not in exclude_cols]
print(f"特征数量: {len(feature_cols)}")

# 划分训练集和测试集
from sklearn.model_selection import train_test_split

X = df[feature_cols]
y = df[target_col]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)

print(f"\n训练集: {X_train.shape}, 样本数: {len(y_train)}, 坏账率: {y_train.mean()*100:.2f}%")
print(f"测试集: {X_test.shape}, 样本数: {len(y_test)}, 坏账率: {y_test.mean()*100:.2f}%")

## Step 3: 特征分箱

使用OptimalBinning对所有特征进行分箱处理。

In [ ]:
from hscredit.core.binning import OptimalBinning
from hscredit.core.metrics import IV

# 分箱配置
n_bins = 6
method = 'best_iv'  # 使用BestIV方法自动选择最优分箱

# 存储分箱结果
binning_results = {}
feature_iv_list = []

print("开始特征分箱...")
for col in feature_cols:
    try:
        binner = OptimalBinning(method=method, max_n_bins=n_bins)
        binner.fit(X_train[[col]], y_train)
        binning_results[col] = binner
        
        # 计算IV值
        iv = IV(X_train[[col]], y_train)
        feature_iv_list.append({'特征': col, 'IV': iv})
    except Exception as e:
        print(f"  特征 {col} 分箱失败: {str(e)}")

print(f"\n完成 {len(binning_results)} 个特征的分箱")

# IV值排序
iv_df = pd.DataFrame(feature_iv_list).sort_values('IV', ascending=False)
print(f"\n特征IV排名（Top 10）:")
print(iv_df.head(10).to_string(index=False))

## Step 4: WOE编码变换

In [ ]:
from hscredit.core.encoders import WOEEncoder

# 创建WOE编码器
woe_encoder = WOEEncoder()

# 对训练集进行WOE编码
X_train_woe = woe_encoder.fit_transform(X_train, y_train)
X_test_woe = woe_encoder.transform(X_test)

print(f"WOE编码后形状: {X_train_woe.shape}")
print(f"\nWOE编码结果（部分）:")
print(X_train_woe.head())

## Step 5: 特征筛选

基于IV值和相关性的双重筛选。

In [ ]:
from hscredit.core.selectors import IVSelector, CorrSelector

# IV筛选
iv_threshold = 0.02
iv_selector = IVSelector(threshold=iv_threshold)
X_train_iv = iv_selector.fit_transform(X_train_woe, y_train)
X_test_iv = iv_selector.transform(X_test_woe)

print(f"IV筛选（阈值>{iv_threshold}）:")
print(f"  筛选前特征数: {X_train_woe.shape[1]}")
print(f"  筛选后特征数: {X_train_iv.shape[1]}")

# 相关性筛选
corr_threshold = 0.8
corr_selector = CorrSelector(threshold=corr_threshold)
X_train_final = corr_selector.fit_transform(X_train_iv, y_train)
X_test_final = corr_selector.transform(X_test_iv)

print(f"\n相关性筛选（阈值>{corr_threshold}）:")
print(f"  筛选前特征数: {X_train_iv.shape[1]}")
print(f"  筛选后特征数: {X_train_final.shape[1]}")

# 最终特征列表
final_features = X_train_final.columns.tolist()
print(f"\n最终入模特征 ({len(final_features)}个):")
for i, f in enumerate(final_features):
    print(f"  {i+1}. {f}")

## Step 6: 模型训练

In [ ]:
from hscredit.core.models import LogisticRegression
from hscredit.core.metrics import KS, AUC, Gini

# 训练逻辑回归模型
lr_model = LogisticRegression(max_iter=1000)
lr_model.fit(X_train_final, y_train)

# 预测
y_train_proba = lr_model.predict_proba(X_train_final)[:, 1]
y_test_proba = lr_model.predict_proba(X_test_final)[:, 1]

# 评估指标
train_ks = KS(y_train, y_train_proba)
train_auc = AUC(y_train, y_train_proba)
train_gini = Gini(y_train, y_train_proba)

test_ks = KS(y_test, y_test_proba)
test_auc = AUC(y_test, y_test_proba)
test_gini = Gini(y_test, y_test_proba)

print("=" * 60)
print("模型性能评估")
print("=" * 60)
print(f"{'指标':<10} {'训练集':<15} {'测试集':<15}")
print("-" * 40)
print(f"{'KS':<10} {train_ks:<15.4f} {test_ks:<15.4f}")
print(f"{'AUC':<10} {train_auc:<15.4f} {test_auc:<15.4f}")
print(f"{'Gini':<10} {train_gini:<15.4f} {test_gini:<15.4f}")

## Step 7: 评分卡生成

In [ ]:
from hscredit.core.models import ScoreCard

# 创建评分卡
scorecard = ScoreCard(
    binner=woe_encoder,
    coef=lr_model.coef_[0],
    intercept=lr_model.intercept_[0],
    pdo=50,       # 每增加50分，odds翻倍
    rate=0.5,     # 基础odds（好坏比1:1）
    score=600,    # 基础分
    floor=300,    # 最低分
    ceiling=800   # 最高分
)

# 生成评分
train_scores = scorecard.predict(X_train_final)
test_scores = scorecard.predict(X_test_final)

print("=" * 60)
print("评分卡参数")
print("=" * 60)
print(f"  基础分(Intercept): 600")
print(f"  PDO(Points to Double Odds): 50")
print(f"  基础Odds: 1:1")
print(f"  分数范围: 300 - 800")

print("\n评分统计:")
print(f"{'数据集':<10} {'最小值':<10} {'最大值':<10} {'均值':<10} {'标准差':<10}")
print("-" * 50)
print(f"{'训练集':<10} {train_scores.min():<10.2f} {train_scores.max():<10.2f} {train_scores.mean():<10.2f} {train_scores.std():<10.2f}")
print(f"{'测试集':<10} {test_scores.min():<10.2f} {test_scores.max():<10.2f} {test_scores.mean():<10.2f} {test_scores.std():<10.2f}")

## Step 8: 分箱明细表

In [ ]:
# 生成分箱明细表
bin_details = []

for col in final_features:
    binner = binning_results[col]
    try:
        bin_table = binner.get_bin_table(col)
        bin_table['特征'] = col
        bin_details.append(bin_table)
    except:
        pass

bin_details_df = pd.concat(bin_details, ignore_index=True)
print(f"分箱明细表形状: {bin_details_df.shape}")
print(f"\n分箱明细表（前20行）:")
print(bin_details_df.head(20).to_string(index=False))

## Step 9: 评分分段分析

In [ ]:
# 评分分段
n_score_bins = 10

def create_score_bins(scores, n_bins=10):
    bins = np.percentile(scores, np.linspace(0, 100, n_bins + 1))
    bins[0] = bins[0] - 1
    bins[-1] = bins[-1] + 1
    return np.digitize(scores, bins[1:-1])

train_score_bins = create_score_bins(train_scores, n_score_bins)
test_score_bins = create_score_bins(test_scores, n_score_bins)

# 统计每个分段的样本数和坏账率
score分段 = []
for i in range(1, n_score_bins + 1):
    train_mask = train_score_bins == i
    test_mask = test_score_bins == i
    
    train_bad_rate = y_train[train_mask].mean() if train_mask.sum() > 0 else np.nan
    test_bad_rate = y_test[test_mask].mean() if test_mask.sum() > 0 else np.nan
    
    score分段.append({
        '评分区间': f'{int(100*(i-1)/n_score_bins)}-{int(100*i/n_score_bins)}分位',
        '训练样本数': train_mask.sum(),
        '训练坏账率': f"{train_bad_rate*100:.2f}%" if not np.isnan(train_bad_rate) else '-',
        '测试样本数': test_mask.sum(),
        '测试坏账率': f"{test_bad_rate*100:.2f}%" if not np.isnan(test_bad_rate) else '-'
    })

score分段_df = pd.DataFrame(score分段)
print("评分分段统计:")
print(score分段_df.to_string(index=False))

## Step 10: 导出评分规则

In [ ]:
# 导出评分规则
rules_list = []

for col in final_features:
    binner = binning_results[col]
    try:
        rules = binner.export_rules()
        for rule in rules:
            rule['特征'] = col
            rule['IV'] = iv_df[iv_df['特征'] == col]['IV'].values[0] if col in iv_df['特征'].values else np.nan
            rules_list.append(rule)
    except:
        pass

rules_df = pd.DataFrame(rules_list)
print(f"导出规则数量: {len(rules_df)}")
print(f"\n规则表结构:")
print(rules_df.columns.tolist())
print(f"\n规则表示例:")
print(rules_df.head(10).to_string(index=False))

## Step 11: 生成Excel完整报告

In [ ]:
from hscredit.report.excel import ExcelWriter

# 创建输出目录
output_dir = '/Users/xiaoxi/CodeBuddy/hscredit/hscredit/examples/scorecard_report'
os.makedirs(output_dir, exist_ok=True)

# 创建Excel报告
report_path = os.path.join(output_dir, '评分卡建模报告.xlsx')
excel_writer = ExcelWriter(report_path)

print("开始生成Excel报告...")

In [ ]:
# Sheet 1: 报告摘要
summary_sheet = excel_writer.get_sheet_by_name("报告摘要")

# 标题
excel_writer.insert_value2sheet(summary_sheet, "B2", "评分卡建模报告", "title")

# 基本信息
info_data = [
    ["项目", "数值"],
    ["训练集样本数", len(y_train)],
    ["测试集样本数", len(y_test)],
    ["入模特征数", len(final_features)],
    ["训练集坏账率", f"{y_train.mean()*100:.2f}%"],
    ["测试集坏账率", f"{y_test.mean()*100:.2f}%"],
]

info_df = pd.DataFrame(info_data[1:], columns=info_data[0])
excel_writer.insert_df2sheet(summary_sheet, info_df, "B4")

# 模型性能
perf_data = [
    ["指标", "训练集", "测试集"],
    ["KS", f"{train_ks:.4f}", f"{test_ks:.4f}"],
    ["AUC", f"{train_auc:.4f}", f"{test_auc:.4f}"],
    ["Gini", f"{train_gini:.4f}", f"{test_gini:.4f}"],
]

perf_df = pd.DataFrame(perf_data[1:], columns=perf_data[0])
excel_writer.insert_df2sheet(summary_sheet, perf_df, "B12")

print("  ✓ 报告摘要")

In [ ]:
# Sheet 2: 特征IV排名
iv_sheet = excel_writer.get_sheet_by_name("特征IV排名")

# 添加标题
excel_writer.insert_value2sheet(iv_sheet, "B2", "特征IV值排名", "header")

# IV排名表
iv_table = iv_df.copy()
iv_table['IV'] = iv_table['IV'].round(4)
iv_table = iv_table.reset_index(drop=True)
iv_table.index = iv_table.index + 1
iv_table.index.name = '排名'

excel_writer.insert_df2sheet(iv_sheet, iv_table, "B4")

print("  ✓ 特征IV排名")

In [ ]:
# Sheet 3: 分箱明细
bin_sheet = excel_writer.get_sheet_by_name("分箱明细")

# 添加标题
excel_writer.insert_value2sheet(bin_sheet, "B2", "特征分箱明细表", "header")

# 分箱明细表
if 'bin_details_df' in dir() and len(bin_details_df) > 0:
    bin_display = bin_details_df.copy()
    excel_writer.insert_df2sheet(bin_sheet, bin_display, "B4")

print("  ✓ 分箱明细")

In [ ]:
# Sheet 4: 评分规则
rules_sheet = excel_writer.get_sheet_by_name("评分规则")

# 添加标题
excel_writer.insert_value2sheet(rules_sheet, "B2", "评分卡规则表", "header")

# 规则表
if 'rules_df' in dir() and len(rules_df) > 0:
    rules_display = rules_df.copy()
    excel_writer.insert_df2sheet(rules_sheet, rules_display, "B4")

print("  ✓ 评分规则")

In [ ]:
# Sheet 5: 评分分段
score_sheet = excel_writer.get_sheet_by_name("评分分段")

# 添加标题
excel_writer.insert_value2sheet(score_sheet, "B2", "评分分段统计", "header")

# 评分分段表
excel_writer.insert_df2sheet(score_sheet, score分段_df, "B4")

print("  ✓ 评分分段")

In [ ]:
# Sheet 6: 模型系数
coef_sheet = excel_writer.get_sheet_by_name("模型系数")

# 添加标题
excel_writer.insert_value2sheet(coef_sheet, "B2", "逻辑回归模型系数", "header")

# 系数表
coef_data = []
for i, col in enumerate(final_features):
    coef_data.append({
        '特征': col,
        '系数': lr_model.coef_[0][i],
        'IV': iv_df[iv_df['特征'] == col]['IV'].values[0] if col in iv_df['特征'].values else np.nan
    })

coef_df = pd.DataFrame(coef_data)
coef_df['系数'] = coef_df['系数'].round(4)
coef_df['IV'] = coef_df['IV'].round(4)
excel_writer.insert_df2sheet(coef_sheet, coef_df, "B4")

# 截距
excel_writer.insert_value2sheet(coef_sheet, "B4", f"截距(Intercept): {lr_model.intercept_[0]:.4f}", "normal")

print("  ✓ 模型系数")

In [ ]:
# Sheet 7: 样本评分
sample_sheet = excel_writer.get_sheet_by_name("样本评分")

# 添加标题
excel_writer.insert_value2sheet(sample_sheet, "B2", "样本评分明细", "header")

# 样本评分表
sample_scores = pd.DataFrame({
    '样本ID': range(len(train_scores)),
    '实际标签': y_train.values,
    '预测概率': y_train_proba.round(4),
    '评分': train_scores.round(2)
})

excel_writer.insert_df2sheet(sample_sheet, sample_scores.head(1000), "B4")

print("  ✓ 样本评分")

In [ ]:
# 保存Excel报告
excel_writer.save()

print("\n" + "=" * 60)
print(f"Excel报告已保存至: {report_path}")
print("=" * 60)

## Step 12: 保存模型

In [ ]:
from hscredit.utils import save_pickle

# 保存模型
save_pickle(lr_model, os.path.join(output_dir, 'lr_model.pkl'))
save_pickle(woe_encoder, os.path.join(output_dir, 'woe_encoder.pkl'))
save_pickle(scorecard, os.path.join(output_dir, 'scorecard.pkl'))
save_pickle(binning_results, os.path.join(output_dir, 'binning_results.pkl'))

print("模型已保存至:")
print(f"  {os.path.join(output_dir, 'lr_model.pkl')}")
print(f"  {os.path.join(output_dir, 'woe_encoder.pkl')}")
print(f"  {os.path.join(output_dir, 'scorecard.pkl')}")
print(f"  {os.path.join(output_dir, 'binning_results.pkl')}")

## 报告输出内容汇总

生成的Excel报告包含以下Sheet:

1. **报告摘要**: 项目基本信息、模型性能指标汇总
2. **特征IV排名**: 所有特征的IV值排序
3. **分箱明细**: 每个特征的分箱边界、WOE值、坏账率
4. **评分规则**: 评分卡规则表达式和分数
5. **评分分段**: 评分分布和分段坏账率
6. **模型系数**: 逻辑回归模型特征系数
7. **样本评分**: 样本ID、实际标签、预测概率、评分

In [ ]:
# 最终输出文件列表
print("\n" + "=" * 60)
print("输出文件汇总")
print("=" * 60)

output_files = os.listdir(output_dir)
for f in output_files:
    filepath = os.path.join(output_dir, f)
    size = os.path.getsize(filepath) / 1024  # KB
    print(f"  {f}: {size:.2f} KB")

print("\n" + "=" * 60)
print("评分卡建模全流程完成！")
print("=" * 60)